In [0]:
#---------------------------------------------------------------------------------#
# DIMENSAO CUSTOMERS                                                              #
#---------------------------------------------------------------------------------#


# Cria dimensao customers conforme especificado
tabela_gold = "zeta_project.gold.dim_customers"

# Cria Dimensão se não existir
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {tabela_gold} (
    customer_id STRING, -- Ajuste o tipo conforme sua Silver
    state STRING,
    city STRING,
    created_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (customer_id)
""")


#Lógica para upsert capturando os campos

spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_silver_customers AS
    SELECT customer_id, state, city, created_ts FROM zeta_project.silver.customers
""")

spark.sql(f"""
MERGE INTO {tabela_gold} AS target
USING vw_silver_customers AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN
  UPDATE SET 
    target.state = source.state,
    target.city = source.city,
    target.created_ts = source.created_ts
WHEN NOT MATCHED THEN
  INSERT (customer_id, state, city, created_ts)
  VALUES (source.customer_id, source.state, source.city, source.created_ts)
""")

In [0]:
#---------------------------------------------------------------------------------#
# DIMENSAO PRODUCTS                                                               #
#---------------------------------------------------------------------------------#


# Cria dimensao customers conforme especificado
tabela_gold = "zeta_project.gold.dim_products"

# Cria Dimensão se não existir
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {tabela_gold} (
    product_id STRING, -- Ajuste o tipo conforme sua Silver
    category STRING,
    brand STRING,
    created_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (product_id)
""")


#Lógica para upsert capturando os campos

spark.sql("""
    CREATE OR REPLACE TEMP VIEW vw_silver_products AS
    SELECT product_id, category, brand, created_ts FROM zeta_project.silver.products
""")

spark.sql(f"""
MERGE INTO {tabela_gold} AS target
USING vw_silver_products AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN
  UPDATE SET 
    target.category = source.category,
    target.brand = source.brand,
    target.created_ts = source.created_ts
WHEN NOT MATCHED THEN
  INSERT (product_id, category, brand, created_ts)
  VALUES (source.product_id, source.category, source.brand, source.created_ts)
""")